# C/GMRES における初期解 $U(0)$ の求め方

|方法|特徴|
|:---|:---|
|解析的に解く|$T(0)=0$で簡単になった最適性条件から、式変形で直接求める|
|Newton-Raphson法|非線形方程式 $F(U)=0$ を反復して解く|
|Newton-GMRES法|Newton-Raphson法の各反復で出る線形方程式をGMRESで解く|
|ホモトピー法・継続法|簡単な問題の解から、ホライゾン長や制約などを徐々に変更し、目的の問題の解へ近づける|

## PMPにおける最適性条件

評価関数を最小化する最適制御問題に対して、Hamiltonian $H$ を用いると、PMPの必要条件は以下のように表される。

$$
\boxed{
\begin{aligned}
\dot{X}(\tau) &= H_\lambda(X(\tau),U(\tau),\lambda(\tau), \tau) \\
\dot{\lambda}(\tau) &= -H_X(X(\tau),U(\tau),\lambda(\tau), \tau) \\
0 &= H_U(X(\tau),U(\tau),\lambda(\tau), \tau) \\
\lambda(T) &= \Phi_X(X(T)) \\
\end{aligned}
}
$$

ここで $\tau$ は予測ホライゾン上の時刻である。

最適制御問題は、予測区間 $\tau \in [0,T]$ において、これらの条件を満たす状態軌道 $X(\tau)$, 制御入力 $U(\tau)$、随伴変数 $\lambda(\tau)$ を求める問題である。

## C/GMRES における初期解

C/GMRESでは、実時刻 $t$ に応じて予測ホライゾン長 $T(t)$ を徐々に伸ばす方法を用いることが出来る。

初期時刻において

$$
T(0) = 0
$$

と設定すると、予測区間の始点と終点が一致するため以下のようになる。

$$
X(T(0)) = X(0)
$$

また、終端条件から以下のように随伴変数が得られる。

$$
\lambda(T(0)) = \Phi_X(X(T(0))) = \Phi_X(X(0))
$$

したがって、初期時刻における停留条件は以下となる。

$$
H_U(X(0), U(0), \Phi_X(X(0)), 0) = 0
$$

ここで初期状態$X(0)$は観測値または設定値として与えられるため、未知数は$U(0)$となる。<br>
したがって、C/GMRESの初期解を求める問題は、この非線形方程式を満たす $U(0)$ を解析的または数値的に求める問題として整理できる。

なお、不等式制約をダミー入力とラグランジュ乗数によって扱う場合は、以下のように制御入力 $u(0)$だけでなく、ダミー入力 $u_d(0)$ や乗数 $\rho(0)$ も含む未知変数ベクトルになる。

$$
U(0) = \begin{bmatrix} u(0) \\ u_d(0) \\ \rho(0) \end{bmatrix}
$$

## 数値的に解く

ここで以下のように残差関数$F$を定義する。

$$
F(U) \equiv H_U(X(0), U, \Phi_X(X(0)), 0)
$$

$X(0)$は与えられるものであるため固定値として捉えると、求めるべき$U$により$F$の値は変化する。そこで

$$F(U)=0$$

となる $U$を求める。

以下では、未知変数ベクトル$U$と残差ベクトル$F(U)$の次元が等しく、$F(U)$のヤコビアン$J_F(U)$が正方行列になる場合を考える。

### Newton-Raphoson法

Newton-Raphson法をベクトル値の非線形方程式 $F(U)=0$ に適用し、初期解 $U(0)$ を求める。

$F(U)$ を $U$ 周りにTaylor展開を行うと、次のようになる。$J_F(U)$は残差関数 $F(U)$のヤコビアンである。

$$
F(U + \Delta U) \approx F(U) + J_{F}(U)\Delta U
$$

適当な$U^{(j)}$を考えたとき、$F(U^{(j)}) \neq 0$ である。そして、少し先の地点が $F(U^{(j)} + \Delta U^{(j)}) = 0$ となる、と考えると、

$$
F(U^{(j)}) + J_{F}(U^{(j)})\Delta U^{(j)} = 0\Rightarrow J_{F}(U^{(j)})\Delta U^{(j)} = -F(U^{(j)}) 
$$

よって、

$$
J_{F}(U^{(j)})\Delta U^{(j)} = -F(U^{(j)})
$$

を $\Delta U^{(j)}$ についてLU分解などで解き、

$$
U^{(j+1)} = U^{(j)} + \Delta U^{(j)}
$$

として更新して、$||F(U^{(j+1)})|| < \delta$ と、$F(U^{(j+1)}) \approx 0$ になるまで繰り返し計算を行う。

ここでヤコビアン $J_F(U)$ は、$F(U) = [f_1(U), f_2(U)]^T$ 、$U=[u_1, u_2]^T$ とすると以下のようになる。

$$
J_F(U) = \begin{bmatrix}
\dfrac{\partial f_1}{\partial u_1} & \dfrac{\partial f_1}{\partial u_2} \\[8pt]
\dfrac{\partial f_2}{\partial u_1} & \dfrac{\partial f_2}{\partial u_2} \\[8pt]
\end{bmatrix}
$$

Newton-Raphson法をまとめると、以下の処理となる。

1. 解の初期候補 $U^{(0)}$ を決め\る
1. $J_{F}(U^{(j)})\Delta U^{(j)} = -F(U^{(j)})$ の式から、LU分解などで $\Delta U^{(j)}$ を求める。
1. $U^{(j+1)} = U^{(j)} + \Delta U^{(j)}$ で解の候補を更新する。
1. $||F(U^{(j+1)})|| < \delta$ と十分に小さくなるまで、2. ,3. を$j=0$から繰り返し行う。
1. $||F(U^{(j+1)})|| < \delta$ となった $U^{(j+1)}$ を$U(0)$とする。


### Newton - GMRES法



Newton-Raphson法では、各反復において次の線形方程式を解く必要がある。

$$
J_F(U^{(j)}) \Delta U^{(j)} = -F(U^{(j)})
$$

これを計算するために、$F(U)$を$U$で偏微分した$J_F(U)$の式が必要である。

$F(U)$の式が複雑な場合、そのヤコビアンも複雑になる可能性がある。また$F(U)$の次元が増えた場合も計算が手間になり導出ミスにつながる。

そこで、[GMRES](./GMRES.ipynb)を用いて $J_F(U^{(j)}) \Delta U^{(j)}= -F(U^{(j)})$を$\Delta U^{(j)}$について解く。

GMRESは$A x = b$という一次連立方程式を、解の初期候補 $x^{(0)}$により$r_0$を計算し、

$$
r_0 = b - A x^{(0)} 
$$

$$
v_1 = \frac{r_0}{\beta} , \quad \beta = ||r_0||
$$

Arnoldi法の繰り返しによりKrynov部分空間の直交基底を広げていく。

Arnoldi法 m 回目の計算は次のようになる。

$$
\begin{aligned}
h_{i,m} &= {v_{i}}^T A v_{m} \space (i=(1, \ 2, \ \cdots \ m)) \\
w &= A v_{m} -\sum_{i=1}^m h_{i,m}v_i \\
h_{m+1, m} &= ||w|| \\
v_{m+1} &= w / h_{m+1, m}
\end{aligned}
$$

この後、Givens回転による上ヘッセンベルグ行列から上三角行列への回転と、右辺の回転を行い、残差の確認を行って小さければ最小化問題より $Y_m$ を計算し、近似解を$x = x^{(0)} + V_m Y_m$のように計算する。  

これを $J_F(U^{(j)}) \Delta U^{(j)} = -F(U^{(j)})$ に当てはめると、$A=J_F(U^{(j)}), x= \Delta U^{(j)}, b = -F(U^{(j)})$ である。

GMRESではヤコビアンを用いずにArnoldi法を実行したい。そのため解の初期候補 $\Delta U_*^{(0)}=0$とすることで、次のように初期残差を計算できる。

$$
\begin{aligned}
r_0 &= -F(U^{(j)}) \\
v_1 &= r_0 / \beta ,\quad \beta = ||r_0||
\end{aligned}
$$

そして、Arnoldi法のなかで計算している$A v_m = J_F(U^{(j)}) v_m$は、$F(U^{(j)} + \varepsilon v_m)$のTaylor展開を用いて

$$
F(U^{(j)} + \varepsilon v_) \approx F(U^{(j)}) + J_F(U^{(j)}) \varepsilon v_m
$$

となるため、以下の有限差分で近似できる。

$$
J_F(U^{(j)}) v_m \approx \frac{F(U^{(j)} + \varepsilon v_m) - F(U^{(j)})}{\varepsilon} 
$$

これを用いて Arnoldi法 m 回目の計算は次のようになる。$m$はArnoldi法の繰り返しの数、$j$はNewton法の繰り返しの数であることに注意する。

$$
\begin{aligned}
Jv &= \frac{F(U^{(j)} + \varepsilon v_m) - F(U^{(j)})}{\varepsilon} \\
h_{i,m} &= {v_{i}}^T J \space (i=(1, \ 2, \ \cdots \ m)) \\
w &= J -\sum_{i=1}^m h_{i,m}v_i \\
h_{m+1, m} &= ||w|| \\
v_{m+1} &= w / h_{m+1, m}
\end{aligned}
$$

この後、Givens回転による上ヘッセンベルグ行列から上三角行列への回転と、右辺の回転を行い、残差の確認を行って小さければ最小化問題より $Y_m$ を計算し、近似解を$\Delta U^{(j)} = V_m Y_m$のように計算する。

そして、この$\Delta U^{(j)}$を用いてNewton法の次の解候補 $U^{(j+1)}$ を計算する。

$$
U^{(j+1)} = U^{(j)} + \Delta U^{(j)}
$$

Newton-GMRES法をまとめると次のようになる。

1. 解の初期候補 $U^{(0)}$ を決め\る
1. $\Delta U^{(j)}$ を$U^{(j)}$を用いてGMRESで求める。
1. $U^{(j+1)} = U^{(j)} + \Delta U$ で解の候補を更新する。
1. $||F(U^{(j+1)})|| < \delta$ と十分に小さくなるまで、2. ,3. を$j=0$から繰り返し行う。
1. $||F(U^{(j+1)})|| < \delta$ となった $U^{(j+1)}$ を$U(0)$とする。


